# Cải thiện truy hồi với "Dense Passage Retrieval"

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepset-ai/haystack/blob/master/tutorials/Tutorial6_Better_Retrieval_via_DPR.ipynb)

### Tầm quan trọng của Retriever

Retriever có ảnh hưởng rất lớn đến hiệu năng của toàn bộ pipeline tìm kiếm.

### Các loại Retriever khác nhau
#### Sparse
Họ thuật toán dựa trên việc đếm tần suất xuất hiện của từ (bag-of-words), tạo ra các vector rất thưa với độ dài bằng kích thước từ vựng.

**Ví dụ**: BM25, TF-IDF

**Ưu điểm**: Đơn giản, nhanh, dễ giải thích

**Nhược điểm**: Phụ thuộc vào việc khớp từ khóa chính xác giữa truy vấn và văn bản

#### Dense
Các retriever này dùng mô hình mạng nơ-ron để tạo vector embedding "dày". Trong nhóm này có hai cách tiếp cận khác nhau:

a) Single encoder: Dùng **một mô hình duy nhất** để nhúng cả truy vấn và đoạn văn.  
b) Dual-encoder: Dùng **hai mô hình**, một để nhúng truy vấn và một để nhúng đoạn văn

Các nghiên cứu gần đây cho thấy dual-encoder hoạt động tốt hơn, có thể vì chúng xử lý tốt hơn sự khác biệt giữa truy vấn và đoạn văn (độ dài, văn phong, cú pháp...).

**Ví dụ**: REALM, DPR, Sentence-Transformers

**Ưu điểm**: Nắm bắt tương đồng ngữ nghĩa thay vì chỉ "khớp từ" (ví dụ: từ đồng nghĩa, chủ đề liên quan...)

**Nhược điểm**: Tốn tính toán hơn, cần huấn luyện mô hình ban đầu

### "Dense Passage Retrieval"

Trong tutorial này, chúng ta sẽ làm nổi bật một "Dense Dual-Encoder" là Dense Passage Retriever.
Nó được giới thiệu bởi Karpukhin và cộng sự (2020, https://arxiv.org/abs/2004.04906).

Tóm tắt gốc:

_"Trả lời câu hỏi miền mở phụ thuộc vào việc truy xuất đoạn văn hiệu quả để chọn ra các ngữ cảnh ứng viên, trong đó các mô hình không gian vectơ thưa thớt truyền thống như TF-IDF hay BM25 là những phương pháp thực tế được dùng phổ biến. Trong công trình này, chúng tôi chỉ ra rằng việc truy xuất hoàn toàn có thể được thực hiện một cách thực tiễn chỉ bằng các biểu diễn dày đặc, với các vector nhúng được học từ một số lượng nhỏ câu hỏi và đoạn văn thông qua một khung mã hóa kép đơn giản. Khi được đánh giá trên nhiều bộ dữ liệu QA miền mở, hệ thống truy xuất dày đặc của chúng tôi vượt trội hơn một hệ thống Lucene-BM25 mạnh với mức cải thiện tuyệt đối từ 9% đến 19% về độ chính xác truy xuất 20 đoạn văn hàng đầu, đồng thời giúp hệ thống QA đầu-cuối của chúng tôi thiết lập các kết quả hiện đại mới trên nhiều điểm chuẩn QA miền mở."_

Bài báo: https://arxiv.org/abs/2004.04906  
Mã nguồn gốc: https://fburl.com/qa-dpr

*Dùng* [liên kết này](https://colab.research.google.com/github/deepset-ai/haystack/blob/master/tutorials/Tutorial6_Better_Retrieval_via_DPR.ipynb) *để mở notebook trong Google Colab.*


### Chuẩn bị môi trường

#### Colab: Bật GPU runtime
Hãy đảm bảo bạn đã bật GPU runtime để tutorial chạy với tốc độ tốt.  
**Runtime -> Change Runtime type -> Hardware accelerator -> GPU**

<img src="https://raw.githubusercontent.com/deepset-ai/haystack/master/docs/img/colab_gpu_runtime.jpg">

In [ ]:
# Make sure you have a GPU running
!nvidia-smi

Fri Apr 15 11:45:15 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla K80           Off  | 00000000:00:04.0 Off |                    0 |
| N/A   35C    P8    28W / 149W |      0MiB / 11441MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
# Install the latest release of Haystack in your own environment
#! pip install farm-haystack

# Install the latest master of Haystack
!pip install --upgrade pip
!pip install git+https://github.com/deepset-ai/haystack.git#egg=farm-haystack[colab,faiss]

In [ ]:
from haystack.utils import clean_wiki_text, convert_files_to_docs, fetch_archive_from_http, print_answers
from haystack.nodes import FARMReader, TransformersReader

INFO - haystack.modeling.model.optimization -  apex not found, won't use it. See https://nvidia.github.io/apex/


### Document Store

#### Tùy chọn 1: FAISS

FAISS là một thư viện để tìm kiếm tương đồng hiệu quả trên tập các vector dày.
`FAISSDocumentStore` sử dụng cơ sở dữ liệu SQL (SQLite in-memory theo mặc định)
ở bên dưới để lưu văn bản tài liệu và các metadata khác. Vector embedding của văn bản
được lập chỉ mục trên FAISS Index để truy vấn khi tìm kiếm câu trả lời.
Biến thể mặc định của `FAISSDocumentStore` là "Flat" nhưng cũng có thể đặt thành "HNSW" để
tìm kiếm nhanh hơn, đánh đổi một phần độ chính xác. Chỉ cần đặt tham số faiss_index_factor_str trong constructor.
Xem thêm về lựa chọn phù hợp cho từng trường hợp: https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index

In [ ]:
from haystack.document_stores import FAISSDocumentStore

document_store = FAISSDocumentStore(faiss_index_factory_str="Flat")

INFO - haystack.telemetry -  Haystack sends anonymous usage data to understand the actual usage and steer dev efforts towards features that are most meaningful to users. You can opt-out at anytime by calling disable_telemetry() or by manually setting the environment variable HAYSTACK_TELEMETRY_ENABLED as described for different operating systems on the documentation page. More information at https://haystack.deepset.ai/guides/telemetry


#### Tùy chọn 2: Milvus

Milvus là một thư viện cơ sở dữ liệu mã nguồn mở cũng được tối ưu cho tìm kiếm tương đồng vector như FAISS.
Tương tự FAISS, Milvus có cả chế độ "Flat" và "HNSW", nhưng vượt trội hơn FAISS ở khả năng quản lý dữ liệu động.
Tuy nhiên, việc thiết lập sẽ phức tạp hơn một chút vì cần chạy qua Docker và cấu hình thêm file.
Xem [tài liệu chính thức](https://milvus.io/docs/v1.0.0/milvus_docker-cpu.md) để biết thêm chi tiết.

In [ ]:
# Milvus cannot be run on COlab, so this cell is commented out.
# To run Milvus you need Docker (versions below 2.0.0) or a docker-compose (versions >= 2.0.0), neither of which is available on Colab.
# See Milvus' documentation for more details: https://milvus.io/docs/install_standalone-docker.md

# !pip install git+https://github.com/deepset-ai/haystack.git#egg=farm-haystack[milvus]

# from haystack.utils import launch_milvus
# from haystack.document_stores import MilvusDocumentStore

# launch_milvus()
# document_store = MilvusDocumentStore()

### Làm sạch và lập chỉ mục tài liệu

Tương tự các tutorial trước, chúng ta sẽ tải xuống, chuyển đổi và lập chỉ mục một số bài viết về Game of Thrones vào DocumentStore.

In [ ]:
# Let's first get some files that we want to use
doc_dir = "data/tutorial6"
s3_url = "https://s3.eu-central-1.amazonaws.com/deepset.ai-farm-qa/datasets/documents/wiki_gameofthrones_txt6.zip"
fetch_archive_from_http(url=s3_url, output_dir=doc_dir)

# Convert files to dicts
docs = convert_files_to_docs(dir_path=doc_dir, clean_func=clean_wiki_text, split_paragraphs=True)

# Now, let's write the dicts containing documents to our DB.
document_store.write_documents(docs)

INFO - haystack.utils.import_utils -  Fetching from https://s3.eu-central-1.amazonaws.com/deepset.ai-farm-qa/datasets/documents/wiki_gameofthrones_txt6.zip to `data/tutorial6`
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/396_Game_of_Thrones__season_6_.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/343_Catelyn_Stark.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/365_A_Song_of_Ice_and_Fire_Roleplaying.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/232_Tommen_Baratheon.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/474_Bronn__character_.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/198_A_Clash_of_Kings.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/359_Kill_the_Boy.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/37_Joffrey_Baratheon.txt
INFO - haystack.utils.preprocessing -  Converting data/tutorial6/78_The_Rains_of_C

Writing Documents:   0%|          | 0/2497 [00:00<?, ?it/s]

### Khởi tạo Retriever, Reader và Pipeline

#### Retriever

**Ở đây:** Chúng ta dùng `DensePassageRetriever`

**Các lựa chọn thay thế:**

- `ElasticsearchRetriever` với truy vấn tùy chỉnh (ví dụ: boosting) và bộ lọc
- Dùng `EmbeddingRetriever` để tìm tài liệu ứng viên dựa trên độ tương đồng embedding (ví dụ: tạo bằng Sentence-BERT)
- Dùng `TfidfRetriever` kết hợp với SQL hoặc InMemory Document Store để prototype và debug đơn giản

In [ ]:
from haystack.nodes import DensePassageRetriever

retriever = DensePassageRetriever(
    document_store=document_store,
    query_embedding_model="facebook/dpr-question_encoder-single-nq-base",
    passage_embedding_model="facebook/dpr-ctx_encoder-single-nq-base",
    max_seq_len_query=64,
    max_seq_len_passage=256,
    batch_size=16,
    use_gpu=True,
    embed_title=True,
    use_fast_tokenizers=True,
)
# Important:
# Now that after we have the DPR initialized, we need to call update_embeddings() to iterate over all
# previously indexed documents and update their embedding representation.
# While this can be a time consuming operation (depending on corpus size), it only needs to be done once.
# At query time, we only need to embed the query and compare it the existing doc embeddings which is very fast.
document_store.update_embeddings(retriever)

INFO - haystack.modeling.utils -  Using devices: CUDA:0
INFO - haystack.modeling.utils -  Number of GPUs: 1


Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/455k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/493 [00:00<?, ?B/s]

INFO - haystack.modeling.model.language_model -  LOADING MODEL
INFO - haystack.modeling.model.language_model -  =============
INFO - haystack.modeling.model.language_model -  Could not find facebook/dpr-question_encoder-single-nq-base locally.
INFO - haystack.modeling.model.language_model -  Looking on Transformers Model Hub (in local cache and online)...


... (7 outputs left out)

Downloading:   0%|          | 0.00/418M [00:00<?, ?B/s]

INFO - haystack.modeling.model.language_model -  Loaded facebook/dpr-ctx_encoder-single-nq-base
INFO - haystack.document_stores.faiss -  Updating embeddings for 2357 docs...


Updating Embedding:   0%|          | 0/2357 [00:00<?, ? docs/s]

Create embeddings:   0%|          | 0/2368 [00:00<?, ? Docs/s]

#### Reader

Tương tự các tutorial trước, bây giờ chúng ta khởi tạo reader.

Ở đây chúng ta dùng FARMReader với mô hình *deepset/roberta-base-squad2* (xem: https://huggingface.co/deepset/roberta-base-squad2)

##### FARMReader

In [ ]:
# Load a  local model or any of the QA models on
# Hugging Face's model hub (https://huggingface.co/models)

reader = FARMReader(model_name_or_path="deepset/roberta-base-squad2", use_gpu=True)

INFO - haystack.modeling.utils -  Using devices: CUDA:0
INFO - haystack.modeling.utils -  Number of GPUs: 1
INFO - haystack.modeling.model.language_model -  LOADING MODEL
INFO - haystack.modeling.model.language_model -  =============
INFO - haystack.modeling.model.language_model -  Could not find deepset/roberta-base-squad2 locally.
INFO - haystack.modeling.model.language_model -  Looking on Transformers Model Hub (in local cache and online)...


Downloading:   0%|          | 0.00/571 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/473M [00:00<?, ?B/s]

INFO - haystack.modeling.model.language_model -  Loaded deepset/roberta-base-squad2
WARNING - haystack.modeling.logger -  Failed to log params: Changing param values is not allowed. Param with key='prediction_heads' was already logged with value='TextSimilarityHead' for run ID='c49f2e4604d04005a98ff0b290ef9ec4'. Attempted logging new value 'QuestionAnsweringHead'.


Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/772 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

WARNING - haystack.modeling.logger -  Failed to log params: Changing param values is not allowed. Param with key='processor' was already logged with value='TextSimilarityProcessor' for run ID='c49f2e4604d04005a98ff0b290ef9ec4'. Attempted logging new value 'SquadProcessor'.
INFO - haystack.modeling.logger -  ML Logging is turned off. No parameters, metrics or artifacts will be logged to MLFlow.
INFO - haystack.modeling.utils -  Using devices: CUDA
INFO - haystack.modeling.utils -  Number of GPUs: 1
INFO - haystack.modeling.infer -  Got ya 2 parallel workers to do inference ...
INFO - haystack.modeling.infer -   0     0  
INFO - haystack.modeling.infer -  /w\   /w\ 
INFO - haystack.modeling.infer -  /'\   / \ 


### Pipeline

Với `Pipeline` của Haystack, bạn có thể ghép các khối xây dựng thành một pipeline tìm kiếm.
Bên dưới, `Pipeline` là các Directed Acyclic Graphs (DAGs) mà bạn có thể dễ dàng tùy chỉnh cho các bài toán riêng.
Để tăng tốc, Haystack cũng cung cấp sẵn một số Pipeline dựng sẵn. Một trong số đó là `ExtractiveQAPipeline`, kết hợp retriever và reader để trả lời câu hỏi.
Bạn có thể tìm hiểu thêm về `Pipeline` trong [docs](https://haystack.deepset.ai/docs/latest/pipelinesmd).

In [ ]:
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(reader, retriever)

## Xong rồi! Hãy đặt câu hỏi!

In [ ]:
# You can configure how many candidates the reader and retriever shall return
# The higher top_k for retriever, the better (but also the slower) your answers.
prediction = pipe.run(
    query="Who created the Dothraki vocabulary?", params={"Retriever": {"top_k": 10}, "Reader": {"top_k": 5}}
)

Inferencing Samples:   0%|          | 0/1 [00:00<?, ? Batches/s]/usr/local/lib/python3.7/dist-packages/haystack/modeling/model/prediction_head.py:478: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  start_indices = flat_sorted_indices // max_seq_len
Inferencing Samples: 100%|██████████| 1/1 [00:00<00:00,  8.93 Batches/s]


In [ ]:
print_answers(prediction, details="minimum")


Query: Who is the father of Arya Stark?
Answers:
[   {   'answer': 'Eddard and Catelyn Stark',
        'context': 'Background ===\n'
                   'Arya is the third child and younger daughter of Eddard and '
                   'Catelyn Stark and is nine years old at the beginning of '
                   'the book series.  Sh'},
    {   'answer': 'Rhaegar',
        'context': ', Aemon Targaryen, Jorah Mormont, Meera Reed, Jon '
                   'Connington and Gilly.\n'
                   'Rhaegar married the Dornish princess Elia Martell of '
                   'Sunspear, and fathered wi'},
    {   'answer': 'Eddard Stark',
        'context': 'e from House Tully in the Riverlands region prior to her '
                   'marriage to Eddard Stark. She has her hair dyed dark brown '
                   'later on while in the Vale, disgui'},
    {   'answer': 'Eddard Stark and Catelyn Stark',
        'context': 'ces==\n'
                   'Sansa Stark is the second child and elde

## Về chúng tôi

Notebook [Haystack](https://github.com/deepset-ai/haystack/) này được tạo bởi [deepset](https://deepset.ai/) tại Berlin, Đức,

Chúng tôi đưa NLP vào công nghiệp thông qua mã nguồn mở!  
Trọng tâm của chúng tôi: các mô hình ngôn ngữ chuyên biệt theo ngành và hệ thống QA quy mô lớn.  
  
Một số dự án khác của chúng tôi:
- [German BERT](https://deepset.ai/german-bert)
- [GermanQuAD and GermanDPR](https://deepset.ai/germanquad)
- [FARM](https://github.com/deepset-ai/FARM)

Liên hệ với chúng tôi:
[Twitter](https://twitter.com/deepset_ai) | [LinkedIn](https://www.linkedin.com/company/deepset-ai/) | [Slack](https://haystack.deepset.ai/community/join) | [GitHub Discussions](https://github.com/deepset-ai/haystack/discussions) | [Website](https://deepset.ai)

Nhân tiện: [chúng tôi đang tuyển dụng!](https://www.deepset.ai/jobs)